In [0]:
import json
import os

# 1. Configuration Constants
COURSE_ID = "py_eng"
SHARED_NOTES_ID = "py_notes_shared"

# 2. Fetch Notes and QA Data from Silver Table
# main_topic_id base meeda order chesthunnam
silver_df = spark.sql(f"SELECT main_topic_id, notes_data, qa_data FROM courseify.default.course_silver WHERE course_id = '{COURSE_ID}' ORDER BY main_topic_id")
silver_records = silver_df.collect()

# 3. Build the Notes Dictionary
notes_dict = {}

for row in silver_records:
    # topic_id ni string ga convert chesthunnam (e.g., "0", "1", "7") endukante JSON keys strings ga undali
    topic_id = str(row['main_topic_id']) 
    
    # Notes data (Markdown text)
    notes_text = row['notes_data'] if row['notes_data'] else ""
    
    # Parse the QA data and rename keys to match your exact required format
    qa_raw = row['qa_data']
    practice_test_formatted = []
    
    if qa_raw:
        try:
            qa_list = json.loads(qa_raw)
            for q_item in qa_list:
                formatted_q = {
                    "q": q_item.get("question", ""),
                    "opt": q_item.get("options", []),
                    "ans_ind": q_item.get("answer_index", 0),
                    "desc": q_item.get("explanation", "")
                }
                practice_test_formatted.append(formatted_q)
        except Exception as e:
            print(f"Error parsing QA JSON for topic {topic_id}: {e}")

    # Assign to the specific topic ID in our dictionary
    notes_dict[topic_id] = {
        "notes": notes_text,
        "practice_test": practice_test_formatted
    }

# 4. Construct the Final Output Structure
final_course_notes_json = {
    "course_notes": {
        SHARED_NOTES_ID: notes_dict
    }
}

# 5. Convert to formatted JSON string and Save to File
output_json_string = json.dumps(final_course_notes_json, indent=4)

file_name = "course_notes_data.json"
with open(file_name, "w", encoding="utf-8") as json_file:
    json_file.write(output_json_string)

# Console Output
current_path = os.path.abspath(file_name)
print(f"✅ Awesome! Notes and Practice Tests JSON Generated Successfully!")
print(f"📁 File saved at: {current_path}\n")

# Just oka chinna preview (optional)
print("Preview of generated keys:")
print(list(final_course_notes_json["course_notes"][SHARED_NOTES_ID].keys()))